In [ ]:
# ── Step 1: Install all libraries
!pip install transformers torch pandas scikit-learn matplotlib networkx feedparser requests -q
!pip install torch_geometric -q
print("✅ All libraries installed")

In [ ]:
# ── Step 2: Imports
import pandas as pd
import numpy as np
import re
import requests
import feedparser
import matplotlib.pyplot as plt
import networkx as nx
from datetime import datetime
import os
print("✅ Imports done")

In [ ]:
# ── Step 3: Fetch REAL news via RSS feeds (no API key needed)
# Real, credible sources → label=0 (REAL)
# Known satire/misinformation sources → label=1 (FAKE/SATIRE)

def fetch_rss(url, source_name, label):
    """Fetch articles from an RSS feed. label: 0=real, 1=fake"""
    try:
        feed = feedparser.parse(url)
        articles = []
        for entry in feed.entries[:25]:
            text = entry.get('summary', '') or entry.get('title', '')
            articles.append({
                'source': source_name,
                'title': entry.get('title', ''),
                'text': text,
                'url': entry.get('link', ''),
                'published': str(entry.get('published', datetime.now())),
                'retweets': 0,
                'true_label': label
            })
        print(f"  ✅ {source_name}: {len(articles)} articles")
        return articles
    except Exception as e:
        print(f"  ⚠️  {source_name} failed: {e}")
        return []

# ── Real news (mainstream/verified sources)
REAL_FEEDS = [
    ('http://feeds.bbci.co.uk/news/rss.xml',             'BBC News'),
    ('https://feeds.reuters.com/reuters/topNews',          'Reuters'),
    ('https://feeds.npr.org/1001/rss.xml',                'NPR'),
    ('https://feeds.apnews.com/rss/apf-topnews',          'AP News'),
    ('https://rss.nytimes.com/services/xml/rss/nyt/HomePage.xml', 'NYT'),
    ('https://www.theguardian.com/world/rss',             'The Guardian'),
]

# ── Satire/misinformation (labeled as fake for training)
FAKE_FEEDS = [
    ('https://www.theonion.com/rss', 'The Onion'),       # satire
    ('https://babylonbee.com/feed',  'Babylon Bee'),     # satire
]

print("Fetching real news articles...")
all_articles = []

for url, name in REAL_FEEDS:
    all_articles.extend(fetch_rss(url, name, label=0))

print("\nFetching satire/fake news articles...")
for url, name in FAKE_FEEDS:
    all_articles.extend(fetch_rss(url, name, label=1))

df_raw = pd.DataFrame(all_articles)
df_raw.to_csv('news_raw.csv', index=False)
print(f"\n📰 Total articles fetched: {len(df_raw)}")
print(f"   Real: {(df_raw.true_label==0).sum()} | Fake/Satire: {(df_raw.true_label==1).sum()}")
df_raw[['source','title','true_label']].head(10)

In [ ]:
# ── Step 3b (OPTIONAL): Also fetch from NewsAPI.org
# Sign up FREE at https://newsapi.org — 100 requests/day
# Replace 'YOUR_KEY_HERE' with your actual key

NEWS_API_KEY = os.environ.get('NEWS_API_KEY', 'YOUR_KEY_HERE')

def fetch_newsapi(api_key, query='misinformation OR fake news', page_size=50):
    if api_key == 'YOUR_KEY_HERE':
        print("⚠️  No NewsAPI key set — skipping. Set env var NEWS_API_KEY to enable.")
        return []
    url = f'https://newsapi.org/v2/everything?q={query}&language=en&pageSize={page_size}&apiKey={api_key}'
    try:
        resp = requests.get(url, timeout=10)
        data = resp.json()
        articles = []
        for a in data.get('articles', []):
            articles.append({
                'source': a['source']['name'],
                'title': a.get('title', ''),
                'text': a.get('description', '') or a.get('title', ''),
                'url': a.get('url', ''),
                'published': a.get('publishedAt', ''),
                'retweets': 0,
                'true_label': -1  # unlabeled — model will predict
            })
        print(f"✅ NewsAPI: {len(articles)} articles")
        return articles
    except Exception as e:
        print(f"NewsAPI error: {e}")
        return []

newsapi_articles = fetch_newsapi(NEWS_API_KEY)
if newsapi_articles:
    df_extra = pd.DataFrame(newsapi_articles)
    df_raw = pd.concat([df_raw, df_extra], ignore_index=True)
    print(f"Total with NewsAPI: {len(df_raw)} articles")
else:
    print("Using RSS data only (still real data!)")

In [ ]:
# ── Step 3c: Load LIAR dataset (real labeled fake news dataset — 12,800 statements)
# This is the gold-standard benchmark for fake news detection
# Source: William Wang 2017, UCSB — publicly available

try:
    from datasets import load_dataset
    print("Loading LIAR dataset from HuggingFace...")
    liar = load_dataset('liar', trust_remote_code=True)
    
    train_data = liar['train']
    liar_rows = []
    # Labels: 0=true, 1=mostly-true, 2=half-true, 3=barely-true, 4=false, 5=pants-fire
    for item in list(train_data)[:500]:  # use 500 samples
        label = 0 if item['label'] <= 1 else 1  # 0=real, 1=fake
        liar_rows.append({
            'source': 'LIAR Dataset',
            'title': item['statement'],
            'text': item['statement'],
            'url': '',
            'published': '',
            'retweets': 0,
            'true_label': label
        })
    
    df_liar = pd.DataFrame(liar_rows)
    df_raw = pd.concat([df_raw, df_liar], ignore_index=True)
    print(f"✅ LIAR dataset: {len(df_liar)} labeled statements added")
    print(f"Total dataset size: {len(df_raw)} items")

except Exception as e:
    print(f"HuggingFace datasets not installed or error: {e}")
    print("Run: !pip install datasets")
    print("Continuing with RSS data only...")

In [ ]:
# ── Step 4: Clean and preprocess text
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\S+', '', text)        # remove URLs
    text = re.sub(r'@\w+|#\w+', '', text)      # remove mentions/hashtags
    text = re.sub(r'[^A-Za-z ]+', '', text)     # keep only letters
    return text.lower().strip()

df = df_raw.copy()
df['clean_text'] = df['text'].apply(clean_text)

# Remove empty rows
df = df[df['clean_text'].str.len() > 10].reset_index(drop=True)

# Add user_id for graph analysis (use source as proxy)
source_map = {s: i+100 for i, s in enumerate(df['source'].unique())}
df['user_id'] = df['source'].map(source_map)

df.to_csv('tweets.csv', index=False)
print(f"✅ Cleaned dataset: {len(df)} articles")
print(f"   Columns: {list(df.columns)}")
df[['source','clean_text','true_label']].head(8)

In [ ]:
# ── Step 5: BERT Text Analysis (fake news detection)
from transformers import pipeline
import torch

print("Loading BERT fake news classifier...")
print("(This downloads ~500MB the first time — will be fast after)")

# Use a pre-trained fake news detector from HuggingFace
# This model was fine-tuned specifically for fake news detection
classifier = pipeline(
    "text-classification",
    model="mrm8488/bert-tiny-finetuned-fake-news-detection",  # lightweight, fast
    device=0 if torch.cuda.is_available() else -1
)

def get_bert_score(text):
    """Returns probability of being FAKE (0=real, 1=fake)"""
    if not text or len(text) < 5:
        return 0.5
    try:
        result = classifier(text[:512])[0]
        # Label mapping: LABEL_1 = fake, LABEL_0 = real
        if result['label'] == 'LABEL_1':
            return result['score']
        else:
            return 1 - result['score']
    except:
        return 0.5

print("Analyzing articles with BERT...")
sample = df.head(50).copy()  # start with 50 — scale up later
sample['text_score'] = sample['clean_text'].apply(get_bert_score)

print("✅ BERT analysis done")
print(sample[['source', 'clean_text', 'text_score', 'true_label']].head(10))

In [ ]:
# ── Step 6: CLIP Image Analysis
# For articles that have an image URL, we download and analyze it
# For text-only articles, we skip image scoring

from transformers import CLIPProcessor, CLIPModel
from PIL import Image
from io import BytesIO
import requests as req
import torch

print("Loading CLIP model...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

FAKE_CUES = [
    "real verified news photo",
    "fake or misleading image",
    "clickbait manipulated photo",
    "edited doctored photograph",
    "genuine authentic news image",
]

def analyze_image_url(image_url):
    """Download image from URL and run CLIP analysis. Returns fake probability."""
    if not image_url or not str(image_url).startswith('http'):
        return None
    try:
        r = req.get(image_url, timeout=5, headers={'User-Agent': 'Mozilla/5.0'})
        if r.status_code != 200 or 'image' not in r.headers.get('content-type',''):
            return None
        image = Image.open(BytesIO(r.content)).convert('RGB')
        inputs = clip_processor(text=FAKE_CUES, images=image, return_tensors='pt', padding=True)
        with torch.no_grad():
            outputs = clip_model(**inputs)
        probs = outputs.logits_per_image.softmax(dim=1)[0]
        # indices 1 and 2 are fake/misleading cues
        fake_prob = (probs[1].item() + probs[2].item()) / 2
        return round(fake_prob, 3)
    except Exception as e:
        return None

print("Running CLIP on articles with image URLs...")
# Only run on subset that has URLs (RSS gives article URLs, not image URLs directly)
# For demo: analyze a few known image URLs
sample['clip_score'] = 0.3  # default (neutral)
print("✅ CLIP analysis ready")
print("Note: Full image analysis runs on articles with direct image URLs")
print("Add image_url column to your data for full CLIP analysis")

In [ ]:
# ── Step 7: Graph / Network Analysis
# Models how content spreads — coordinated sharing = suspicious
import networkx as nx
import numpy as np

G = nx.Graph()

# Build graph: nodes=sources, edges=shared keywords (coordinated spread signal)
sources = df['source'].unique()
G.add_nodes_from(sources)

# Connect sources that share similar keywords (simplified co-citation graph)
from collections import Counter

def top_keywords(texts, n=10):
    words = ' '.join(texts).split()
    common = [w for w, _ in Counter(words).most_common(50) if len(w) > 4]
    return set(common[:n])

source_keywords = {}
for source in sources:
    texts = df[df['source']==source]['clean_text'].tolist()
    source_keywords[source] = top_keywords(texts)

# Add edges where sources share keywords (potential coordination)
for i, s1 in enumerate(sources):
    for s2 in sources[i+1:]:
        shared = len(source_keywords[s1] & source_keywords[s2])
        if shared > 3:
            G.add_edge(s1, s2, weight=shared)

# Compute graph scores — high centrality = potential coordinated spread
centrality = nx.degree_centrality(G)

def graph_score(source):
    """Higher score = more connected = more suspicious spreading pattern"""
    return round(centrality.get(source, 0.0), 3)

df['graph_score'] = df['source'].apply(graph_score)

# Visualize the network
plt.figure(figsize=(10, 7))
pos = nx.spring_layout(G, seed=42)
node_colors = ['#e74c3c' if 'Onion' in n or 'Bee' in n else '#3498db' for n in G.nodes()]
nx.draw_networkx(G, pos, node_color=node_colors, node_size=800,
                 font_size=8, edge_color='gray', alpha=0.8)
plt.title("TruthLens Network Graph: Source Coordination", fontsize=13)
plt.axis('off')
plt.tight_layout()
plt.savefig('network_graph.png', dpi=150)
plt.show()
print("✅ Graph analysis done")
print(df[['source','graph_score']].drop_duplicates().head(10))

In [ ]:
# ── Step 8: Combined TruthLens Score
# Weighted combination of all signals

df['final_score'] = (
    0.55 * df['text_score'] +           # BERT text analysis (most important)
    0.30 * df['graph_score'] +          # network/spread pattern
    0.15 * df['clip_score']             # image analysis
)

df['predicted_label'] = (df['final_score'] > 0.5).astype(int)

print("✅ TruthLens final scores calculated")
print()
print("Sample predictions:")
display_cols = ['source', 'title', 'text_score', 'graph_score', 'final_score', 'predicted_label']
print(df[display_cols].head(15).to_string(index=False))

In [ ]:
# ── Step 9: Evaluate on labeled data
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import matplotlib.pyplot as plt

# Only evaluate on rows with ground-truth labels (true_label != -1)
eval_df = df[df['true_label'].isin([0, 1])].copy()

if len(eval_df) > 5:
    y_true = eval_df['true_label']
    y_pred = eval_df['predicted_label']
    
    accuracy  = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall    = recall_score(y_true, y_pred, zero_division=0)
    f1        = f1_score(y_true, y_pred, zero_division=0)
    
    print(f"Evaluated on {len(eval_df)} labeled articles\n")
    print(f"Accuracy:  {accuracy:.3f}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall:    {recall:.3f}")
    print(f"F1 Score:  {f1:.3f}")
    print()
    print(classification_report(y_true, y_pred, target_names=['Real','Fake']))
    
    # Bar chart
    metrics  = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    values   = [accuracy, precision, recall, f1]
    plt.figure(figsize=(6, 4))
    bars = plt.bar(metrics, values, color=['#3498db','#2ecc71','#e67e22','#9b59b6'])
    plt.ylim(0, 1.1)
    plt.title('TruthLens Performance Evaluation')
    plt.ylabel('Score')
    for bar, val in zip(bars, values):
        plt.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.2f}', ha='center', fontsize=11)
    plt.tight_layout()
    plt.savefig('performance_metrics.png', dpi=150)
    plt.show()
else:
    print("Not enough labeled data for evaluation")
    print("Use the LIAR dataset (Step 3c) to get labeled data")

In [ ]:
# ── Step 10: Distribution Charts
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Chart 1: Fake vs Real count
counts = df['predicted_label'].value_counts()
axes[0].bar(['Real News', 'Fake News'],
            [counts.get(0, 0), counts.get(1, 0)],
            color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Fake vs Real News Distribution')
axes[0].set_ylabel('Number of Articles')
for i, v in enumerate([counts.get(0,0), counts.get(1,0)]):
    axes[0].text(i, v + 0.3, str(v), ha='center', fontsize=12)

# Chart 2: Score distribution by source
source_avg = df.groupby('source')['final_score'].mean().sort_values(ascending=False)
axes[1].barh(source_avg.index, source_avg.values, color='#3498db')
axes[1].set_title('Average Fake Score by Source')
axes[1].set_xlabel('Fake Probability (higher = more suspicious)')
axes[1].axvline(0.5, color='red', linestyle='--', alpha=0.7, label='Decision boundary')
axes[1].legend()

plt.tight_layout()
plt.savefig('distributions.png', dpi=150)
plt.show()
print("✅ Charts saved")

In [ ]:
# ── Step 11: Latency Analysis
import time

latencies = []
test_texts = df['clean_text'].dropna().head(5).tolist()

for text in test_texts:
    start = time.time()
    get_bert_score(text)
    latencies.append(time.time() - start)

plt.figure(figsize=(6, 4))
plt.plot(range(1, len(latencies)+1), latencies, marker='o', color='#3498db')
plt.xlabel('Run Number')
plt.ylabel('Time (seconds)')
plt.title('TruthLens Inference Latency')
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('latency.png', dpi=150)
plt.show()
print(f"Average latency: {np.mean(latencies)*1000:.1f} ms per article")

In [ ]:
# ── Step 12: Save All Results
import json

# Save predictions
df.to_csv('final_predictions.csv', index=False)

# Save metrics
if 'accuracy' in dir():
    metrics_dict = {
        'accuracy': round(accuracy, 4),
        'precision': round(precision, 4),
        'recall': round(recall, 4),
        'f1_score': round(f1, 4),
        'total_articles': len(df),
        'real_count': int((df.predicted_label==0).sum()),
        'fake_count': int((df.predicted_label==1).sum()),
    }
    with open('metrics.json', 'w') as f:
        json.dump(metrics_dict, f, indent=4)
    print("✅ metrics.json saved")

print("✅ final_predictions.csv saved")
print(f"\nTruthLens analysis complete!")
print(f"Analyzed {len(df)} real articles from {df['source'].nunique()} sources")
print(f"Detected {int((df.predicted_label==1).sum())} potentially fake/misleading articles")